# VisionQ — Fase II: Ejecución en Google Colab

**Gestión de proyectos de inteligencia artificial · Proyecto VisionQ**

Este cuaderno reproduce, de principio a fin, la solución desplegable de VisionQ:

1. Clona el repositorio de GitHub e instala dependencias en un entorno aislado.
2. Ejecuta la batería de **46 pruebas automatizadas** (funcionales, casos extremos, calidad del modelo).
3. Levanta la **API FastAPI** (backend) y el **frontend** que la consume.
4. Demuestra la **integración por API**: endpoints de salud, inspección visual (individual y por lote) y predicción de fallas.
5. Evalúa **casos extremos**, ejecuta la **prueba de humo E2E** y una **prueba de carga**.
6. Abre el **frontend** dentro de Colab.
7. (Opcional) Reproduce la **evaluación de robustez** del modelo con el dataset NEU-DET completo.

> **Nota sobre Docker:** Colab no permite ejecutar el daemon de Docker. Aquí se ejecuta exactamente el mismo código y los mismos modelos que empaqueta el `Dockerfile`, servidos con Uvicorn. La imagen Docker se construye y valida localmente (`docker build`) o en la nube (Cloud Build / Cloud Run); ver la sección 9.

**Entorno recomendado:** Runtime de CPU estándar (no se requiere GPU).

## 1. Preparación del entorno

In [ ]:
import os, sys, subprocess, platform, json, time, io
from pathlib import Path

REPO_URL = "https://github.com/edgdark80-lgtm/VisionQ.git"
BRANCH   = "main"
IN_COLAB = "google.colab" in sys.modules
BASE     = Path("/content") if IN_COLAB else Path.cwd()
REPO     = Path(os.environ.get("VISIONQ_REPO_DIR", BASE / "VisionQ"))
VENV     = BASE / "vqenv"
PORT     = 8000
API      = f"http://127.0.0.1:{PORT}"

print("Python del cuaderno:", platform.python_version(), "| Colab:", IN_COLAB)
print("Repositorio:", REPO)

In [ ]:
def sh(cmd, cwd=None, check=True):
    """Ejecuta un comando de shell mostrando su salida en vivo."""
    print(f"$ {cmd}")
    env = {**os.environ, "MPLBACKEND": "Agg"}   # evita heredar el backend gráfico del kernel
    p = subprocess.run(cmd, shell=True, cwd=cwd, text=True, env=env,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout[-6000:])
    if check and p.returncode != 0:
        raise RuntimeError(f"Falló (código {p.returncode}): {cmd}")
    return p

import shutil

if "VISIONQ_REPO_DIR" in os.environ and (REPO / "app" / "main.py").exists():
    print("Usando copia local del repositorio en", REPO)
elif (REPO / ".git").exists():
    # Ya existe un clon (p. ej., de una ejecución anterior): se sincroniza con la última versión de GitHub
    sh(f"git -C {REPO} fetch --depth 1 origin {BRANCH}")
    sh(f"git -C {REPO} reset --hard origin/{BRANCH}")
else:
    if REPO.exists():
        shutil.rmtree(REPO)   # carpeta incompleta o ajena: se elimina para clonar limpio
    sh(f"git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO}")

assert (REPO / "app" / "main.py").exists(), "El clon no contiene el código de VisionQ; revisa REPO_URL/BRANCH"
sh(f"git -C {REPO} log --oneline -1")
sh("ls", cwd=REPO)

### 1.1 Dependencias en un entorno virtual aislado
Se instalan las versiones fijadas en `requirements.txt` (las mismas del contenedor) dentro de un entorno virtual,
para no alterar las librerías preinstaladas de Colab ni obligar a reiniciar el runtime.

In [ ]:
PY = VENV / "bin" / "python"
if not PY.exists():
    sh(f"{sys.executable} -m pip install -q virtualenv")
    sh(f"{sys.executable} -m virtualenv -q {VENV}")
sh(f"{PY} -m pip install -q --upgrade pip")
sh(f"{PY} -m pip install -q -r requirements-dev.txt", cwd=REPO)
sh(f"{PY} -c \"import fastapi, sklearn, skimage; print('fastapi', fastapi.__version__, '| scikit-learn', sklearn.__version__, '| scikit-image', skimage.__version__)\"")

## 2. Pruebas automatizadas (pytest)
Incluye pruebas funcionales de cada endpoint, casos extremos (archivos vacíos, corruptos, MIME no soportado,
imágenes negras/blancas/ruido, valores físicamente imposibles, lotes vacíos o excesivos, API key) y umbrales mínimos de calidad del modelo.

In [ ]:
r = sh(f"{PY} -m pytest -q -p no:cacheprovider", cwd=REPO, check=False)
print("RESULTADO:", "TODAS LAS PRUEBAS PASARON" if r.returncode == 0 else "HAY PRUEBAS FALLIDAS")

## 3. Levantar la API (backend + frontend)

In [ ]:
import urllib.request

def wait_health(url, timeout=60):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            with urllib.request.urlopen(url + "/api/v1/health", timeout=2) as r:
                if r.status == 200:
                    return json.loads(r.read())
        except Exception:
            time.sleep(1)
    raise TimeoutError("La API no respondió a tiempo")

subprocess.run(f"pkill -f 'uvicorn app.main:app' || true", shell=True)
env = {**os.environ, "VISIONQ_ENVIRONMENT": "colab", "PORT": str(PORT)}
log = open(BASE / "visionq_server.log", "w")
server = subprocess.Popen([str(VENV / "bin" / "uvicorn"), "app.main:app", "--host", "0.0.0.0", "--port", str(PORT)],
                          cwd=REPO, env=env, stdout=log, stderr=subprocess.STDOUT)
health = wait_health(API)
print(json.dumps(health, indent=2, ensure_ascii=False))

## 4. Integración por API — endpoints de sistema

In [ ]:
import requests, pandas as pd

for ep in ["/api/v1/health", "/api/v1/models"]:
    r = requests.get(API + ep)
    print(f"GET {ep} -> {r.status_code}  X-Request-ID={r.headers.get('X-Request-ID')}")
    print(json.dumps(r.json(), indent=2, ensure_ascii=False)[:1500], "\n")

spec = requests.get(API + "/openapi.json").json()
pd.DataFrame([{"método": m.upper(), "ruta": p, "resumen": d.get("summary", "")}
              for p, ops in spec["paths"].items() for m, d in ops.items()])

## 5. Inspección visual de defectos
Se envían a `POST /api/v1/inspect` las 30 imágenes de `samples/` (5 por clase, tomadas del conjunto de validación externo de NEU-DET,
no usado en el entrenamiento) y se compara la predicción con la etiqueta real (prefijo del archivo).

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

samples = sorted((REPO / "samples").glob("*.jpg"))
rows = []
for f in samples:
    with open(f, "rb") as fh:
        r = requests.post(API + "/api/v1/inspect", files={"file": (f.name, fh, "image/jpeg")})
    j = r.json()
    rows.append({"archivo": f.name, "real": f.name.rsplit("_", 1)[0], "predicho": j["defect_class"],
                 "confianza": j["confidence"], "decisión": j["decision"], "OOD": j["out_of_distribution"],
                 "latencia_ms": j["latency_ms"]})
df = pd.DataFrame(rows)
df["correcto"] = df.real == df.predicho
print(f"Exactitud en 30 imágenes externas: {df.correcto.mean():.1%} | latencia media {df.latencia_ms.mean():.0f} ms")
df

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(15, 8))
for ax, (_, r) in zip(axes.flat, df.groupby("real").head(3).iterrows()):
    ax.imshow(Image.open(REPO / "samples" / r.archivo), cmap="gray"); ax.axis("off")
    color = "#2E7D32" if r.correcto else "#B3261E"
    ax.set_title(f"real: {r.real}\npred: {r.predicho} ({r.confianza:.2f})\n{r.decisión}", fontsize=8, color=color)
plt.suptitle("VisionQ · respuestas de /api/v1/inspect", fontsize=12); plt.tight_layout(); plt.show()

In [ ]:
# Inspección por lote (hasta 16 imágenes por petición)
files = [("files", (f.name, open(f, "rb"), "image/jpeg")) for f in samples[:6]]
r = requests.post(API + "/api/v1/inspect/batch", files=files)
print("POST /api/v1/inspect/batch ->", r.status_code)
pd.DataFrame([{k: it.get(k) for k in ("filename", "defect_class", "confidence", "decision")} for it in r.json()["results"]])

## 6. Predicción de fallas de máquina

In [ ]:
normal = {"type": "M", "air_temperature_k": 298.1, "process_temperature_k": 308.6,
          "rotational_speed_rpm": 1551, "torque_nm": 42.8, "tool_wear_min": 0}
riesgo = {"type": "L", "air_temperature_k": 302.0, "process_temperature_k": 310.5,
          "rotational_speed_rpm": 1300, "torque_nm": 60.0, "tool_wear_min": 210}
for name, body in [("normal", normal), ("riesgo", riesgo)]:
    r = requests.post(API + "/api/v1/failure/predict", json=body)
    print(f"[{name}] HTTP {r.status_code}")
    print(json.dumps(r.json(), indent=2, ensure_ascii=False), "\n")

In [ ]:
# Sensibilidad: probabilidad de falla vs. desgaste de herramienta (lote de 25 lecturas en una sola petición)
wear = list(range(0, 250, 10))
batch = {"readings": [{**riesgo, "tool_wear_min": w, "torque_nm": 45.0, "rotational_speed_rpm": 1500,
                      "air_temperature_k": 300.0, "process_temperature_k": 310.5} for w in wear]}
r = requests.post(API + "/api/v1/failure/predict/batch", json=batch)
probs = [p["failure_probability"] for p in r.json()["results"]]
thr = requests.get(API + "/api/v1/models").json()["failure"]["threshold"]
plt.figure(figsize=(8, 3.5)); plt.plot(wear, probs, marker="o"); plt.axhline(thr, ls="--", color="#B3261E", label=f"umbral {thr}")
plt.xlabel("Desgaste de herramienta (min)"); plt.ylabel("P(falla)"); plt.legend(); plt.title("Respuesta del modelo al desgaste"); plt.show()

## 7. Casos extremos contra la API en ejecución
Se verifica que la API **rechace entradas inválidas con el código HTTP correcto** y que el modelo **no tome decisiones automáticas**
sobre imágenes que no son de acero o con captura deficiente (responde `REVISION_MANUAL`).

In [ ]:
import numpy as np

def png_bytes(arr):
    b = io.BytesIO(); Image.fromarray(arr.astype("uint8")).save(b, "PNG"); return b.getvalue()

rng = np.random.default_rng(0)
yy, xx = np.mgrid[0:200, 0:200]
casos = {
    "Archivo vacío":            (b"", "image/png", None),
    "Bytes corruptos":          (b"\x89PNG esto no es una imagen", "image/png", None),
    "Texto plano (MIME)":       (b"hola", "text/plain", None),
    "Imagen negra":             (png_bytes(np.zeros((200, 200))), "image/png", None),
    "Imagen blanca":            (png_bytes(np.full((200, 200), 255)), "image/png", None),
    "Ruido uniforme":           (png_bytes(rng.integers(0, 256, (200, 200))), "image/png", None),
    "Tablero de ajedrez":       (png_bytes(((xx // 25 + yy // 25) % 2) * 255), "image/png", None),
    "Imagen diminuta 8x8":      (png_bytes(rng.integers(80, 160, (8, 8))), "image/png", None),
    "Muestra real RGB 1200x900": (png_bytes(np.array(Image.open(REPO / "samples/scratches_195.jpg").convert("RGB").resize((1200, 900)))), "image/png", None),
    "Ruido RGB 1200x900":        (png_bytes(rng.integers(0, 256, (900, 1200, 3))), "image/png", None),
}
out = []
for nombre, (data, mime, _) in casos.items():
    r = requests.post(API + "/api/v1/inspect", files={"file": ("caso.png", data, mime)})
    j = r.json()
    out.append({"caso": nombre, "HTTP": r.status_code, "decisión": j.get("decision", "—"),
                "OOD": j.get("out_of_distribution", "—"), "detalle": (j.get("decision_reason") or j.get("error") or "")})
pd.set_option("display.max_colwidth", 90)
pd.DataFrame(out)
# Nota: el ruido RGB de alta resolución, al reducirse a 200x200, se suaviza y adquiere textura parecida al acero;
# es una limitación conocida del detector OOD documentada en el informe de validación.

In [ ]:
casos_f = {
    "Velocidad negativa":           {**normal, "rotational_speed_rpm": -50},
    "Tipo inválido 'X'":            {**normal, "type": "X"},
    "Temperatura proceso < aire":   {**normal, "process_temperature_k": 290.0},
    "Campo faltante":               {k: v for k, v in normal.items() if k != "torque_nm"},
    "Extrapolación (par 95 Nm)":    {**normal, "torque_nm": 95.0},
}
out = []
for nombre, body in casos_f.items():
    r = requests.post(API + "/api/v1/failure/predict", json=body)
    j = r.json()
    out.append({"caso": nombre, "HTTP": r.status_code,
                "resultado": j.get("risk_level") or j.get("error"),
                "detalle": "; ".join(d["mensaje"] for d in j["detail"]) if isinstance(j.get("detail"), list) else (j.get("detail") or ""),
                "fuera_de_rango": j.get("out_of_training_range", "")})
r = requests.post(API + "/api/v1/failure/predict", data="{no es json", headers={"Content-Type": "application/json"})
out.append({"caso": "JSON malformado", "HTTP": r.status_code, "resultado": r.json().get("error"), "detalle": "", "fuera_de_rango": ""})
pd.DataFrame(out)

## 8. Prueba de humo E2E y prueba de carga

In [ ]:
r = sh(f"bash scripts/smoke_test.sh {API}", cwd=REPO, check=False)

In [ ]:
r = sh(f"{PY} scripts/load_test.py --base {API} --n 40 --concurrency 1 4", cwd=REPO, check=False)

In [ ]:
m = requests.get(API + "/api/v1/metrics").json()
print(json.dumps(m, indent=2, ensure_ascii=False))

## 9. Frontend dentro de Colab
La celda siguiente publica el puerto 8000 a través del proxy de Colab y abre la interfaz web (`/ui/`),
que consume la misma API (pestañas *Inspección visual*, *Predicción de fallas* y *Sistema*).
Fuera de Colab, abre `http://localhost:8000/ui/` en el navegador.

In [ ]:
if IN_COLAB:
    from google.colab import output
    output.serve_kernel_port_as_window(PORT, path="/ui/")
    print("Swagger UI:")
    output.serve_kernel_port_as_window(PORT, path="/docs")
else:
    print(f"Frontend: {API}/ui/   |   Swagger UI: {API}/docs")

In [ ]:
# Vista embebida del frontend dentro de la salida de la celda (solo Colab)
if IN_COLAB:
    from google.colab import output
    output.serve_kernel_port_as_iframe(PORT, path="/ui/", height=900)

### 9.1 Contenedor Docker
El `Dockerfile` del repositorio empaqueta exactamente este backend, el frontend y los modelos. En una máquina con Docker:

```bash
docker build -t visionq:2.0.0 .
docker run -d --name visionq -p 8000:8000 --read-only --tmpfs /tmp visionq:2.0.0
bash scripts/smoke_test.sh http://localhost:8000
```

Para desplegar en la nube sin Docker local (p. ej., desde Colab con `gcloud`), Cloud Build construye la imagen de forma remota:

```bash
PROJECT_ID=mi-proyecto bash deploy/gcp-cloud-run.sh
```

In [ ]:
print((REPO / "Dockerfile").read_text())

## 10. (Opcional) Evaluación de robustez con NEU-DET completo
Descarga las 1 770 imágenes de NEU-DET (~30 MB) y ejecuta `scripts/evaluate_models.py`: perturbaciones (ruido, desenfoque,
brillo, contraste, rotación, compresión JPEG), entradas fuera de distribución, barrido de umbral de confianza y recall por modo de falla.
Tarda ~3–5 minutos en CPU.

In [ ]:
RUN_FULL_EVAL = True   # cambia a False para omitir

if RUN_FULL_EVAL:
    if not (REPO / "data/neu_det/IMAGES").exists():
        sh("bash training/download_data.sh", cwd=REPO)
    sh(f"{PY} -m pip install -q matplotlib")
    r = sh(f"{PY} scripts/evaluate_models.py", cwd=REPO, check=False)

In [ ]:
res = json.loads((REPO / "docs/evidence/evaluation_results.json").read_text())
pert = pd.DataFrame(res["vision_perturbations"]).T[["accuracy", "manual_review_rate", "ood_rate", "mean_confidence"]]
display(pert.style.format("{:.3f}").background_gradient(subset=["accuracy"], cmap="RdYlGn", vmin=0.6, vmax=1.0))
print("Detección de entradas fuera de distribución:", res["vision_ood_detection_rate"])
pd.DataFrame(res["failure_recall_by_mode"]).T

In [ ]:
from IPython.display import Image as IPImage, display
for png in ["chart_vision_confusion.png", "chart_vision_robustness.png", "chart_vision_threshold.png",
            "chart_failure_pr.png", "chart_failure_sensitivity.png"]:
    p = REPO / "docs/evidence" / png
    if p.exists():
        display(IPImage(filename=str(p), width=720))

## 11. Cierre

In [ ]:
server.terminate(); server.wait(timeout=10)
print("Servidor detenido. Últimas líneas del log del servidor:")
print("".join(open(BASE / "visionq_server.log").readlines()[-5:]))

**Conclusión.** En Colab se comprobó el funcionamiento integral de VisionQ: las pruebas automatizadas pasan, la API expone y documenta
sus endpoints (OpenAPI), el frontend consume la API, las entradas inválidas se rechazan con códigos HTTP correctos y el modelo deriva a
revisión manual los casos fuera de distribución. La misma aplicación se distribuye como imagen Docker para su despliegue en Cloud Run,
Render o Azure Container Apps (ver `docs/Manual_Despliegue_VisionQ.docx`).

**Referencias**

Matzka, S. (2020). *AI4I 2020 predictive maintenance dataset* [Data set]. UCI Machine Learning Repository. https://doi.org/10.24432/C5HS5C

Song, K., & Yan, Y. (2013). A noise robust method based on completed local binary patterns for hot-rolled steel strip surface defects. *Applied Surface Science, 285*, 858–864. https://doi.org/10.1016/j.apsusc.2013.09.002